# Multiple Knapsack Problem

\begin{align*}
\text{Maximize} \quad & \sum_{i=1}^{N} \sum_{k=1}^{K} v_i x_{ik} & \\
\text{Subject to} \quad & \sum_{k=1}^{K} x_{ik} \leq 1 & (i = 1, 2, \dots, N) \\
& \sum_{i=1}^{N} w_i x_{ik} \leq C_k & (k = 1, 2, \dots, K) \\
& x_{ik} \in \{0, 1\} & (i = 1, 2, \dots, N,\ k = 1, 2, \dots, K)
\end{align*}

### Parameters
- $N$ and $K$ are the number of items and knapsacks respectively.
- $v_i$ is the value of item $i$.
- $w_i$ is the weight of item $i$.
- $C_k$ is the capacity of knapsack $k$.

### Decision Variables
- $x_{ik}$: equals 1 if item $i$ is placed in knapsack $k$, 0 otherwise.




###0. Preprocessing:
Set your Gurobi license, install `gurobipy`, and define the environment

In [ ]:
!pip install gurobipy
import gurobipy as gp
from gurobipy import GRB, quicksum

# WLS Academic License — replace with your own credentials
# env = gp.Env(params={
#     'WLSACCESSID': 'your-access-id',
#     'WLSSECRET':   'your-secret',
#     'LICENSEID':    0000000
# })

###1. Import Libraries

In [ ]:
import numpy as np   # for random number generation
import time          # for tracking solution time

###2. Parameter Settings

Set the number of items and knapsacks. These values can be changed to test other versions of the problem.

In [ ]:
rnd = np.random
rnd.seed(42)  # Fixed seed ensures reproducible results

In [ ]:
# ── Change these two values to run other versions of the problem ──
N = 6   # number of items
K = 3   # number of knapsacks

Create index lists for items and knapsacks.

In [ ]:
I = [i for i in range(1, N + 1)]   # item indices: {1, 2, ..., N}
J = [k for k in range(1, K + 1)]   # knapsack indices: {1, 2, ..., K}

Randomly generate values and weights for each item.

**Feasibility guarantee:** item weights are drawn from $[5, 20]$ and knapsack capacities from $[30, 60]$, so every single item fits in any knapsack. Total capacity is also guaranteed to exceed total weight.

In [ ]:
v = {i: rnd.randint(10, 50) for i in I}   # v_i: value of item i
w = {i: rnd.randint(5,  20) for i in I}   # w_i: weight of item i

Randomly generating capacities for all knapsacks.

In [ ]:
C = {k: rnd.randint(30, 60) for k in J}   # C_k: capacity of knapsack k

Making sure that Total Capacity >= Total Weight.

In [ ]:
# If total capacity is less than total weight, increase the last knapsack's capacity
if sum(C.values()) < sum(w.values()):
    diff = sum(w.values()) - sum(C.values())
    C[K] += diff   # add the deficit to the last knapsack

###4. Model Preparation

Model Initialization

In [ ]:
model_mk = gp.Model('Multiple_Knapsack')   # create a new Gurobi model

Adding decision variables

In [ ]:
# x[i, k] = 1 if item i is placed in knapsack k, 0 otherwise
x = model_mk.addVars(I, J, vtype=GRB.BINARY, name="x")

Adding assignment constraints — each item can be assigned to at most one knapsack

In [ ]:
# Each item i can go into at most one knapsack
for i in I:
    model_mk.addConstr(quicksum(x[i, k] for k in J) <= 1, name="Assign(%s)" % i)

Adding capacity constraints — total weight in each knapsack cannot exceed its capacity

In [ ]:
# Total weight of items placed in knapsack k must not exceed its capacity C_k
model_mk.addConstrs((quicksum(w[i] * x[i, k] for i in I) <= C[k] for k in J), name="Capacity")

Adding the objective function:

In [ ]:
# Maximize total value of all items placed across all knapsacks
model_mk.setObjective(quicksum(v[i] * x[i, k] for i in I for k in J), GRB.MAXIMIZE)

Update and write the model

In [ ]:
model_mk.update()                          # finalize model structure
model_mk.write('Multiple_Knapsack.lp')     # export model to .lp file

###5. Model Solution
Solving the model and printing the solution

In [ ]:
start = time.time()    # record start time
model_mk.optimize()    # run the Gurobi solver
end = time.time()      # record end time

In [ ]:
def print_solution_mk(model, x, v, w, C, I, J):
    """Print the optimal solution: objective value, solve time, and items per knapsack."""
    if model.status != GRB.OPTIMAL:
        print("Model is not Optimized")
    else:
        print('Objective function value: ', model.objVal)
        print('Solution time: ', end - start, ' seconds')
        print('=================================================')
        for k in J:
            # collect items assigned to knapsack k
            items_in_k = [i for i in I if x[i, k].x > 1e-6]
            load = sum(w[i] for i in items_in_k)   # total weight used
            print(f"Knapsack {k} (Capacity: {C[k]}, Load: {load}):")
            for i in items_in_k:
                print(f"  x[{i},{k}] = 1  |  Value: {v[i]}, Weight: {w[i]}")
            if not items_in_k:
                print("  (empty)")

In [ ]:
print_solution_mk(model_mk, x, v, w, C, I, J)